# Protótipo de busca dirigida - simple_search.py

A ideia do colab é explicar o funcionamento para alguém que nunca programou em Python, sem esconder a lógica dentro de bibliotecas.

O fluxo que quero demonstrar é:

**Pergunta → Recuperação → Síntese → Evidências → Validação humana**

Dê "play" nos blocos de código do Collab pra acompanhar a execução passo a passo. Lembrando que o fluxo de funcionamento do notebook exige que o bloco anterior seja executado, caso contrário dará erro por uma função ou variável não ter sido definida. Para tal, basta clicar no botão de "play" que está disponível nesses blocos. Alguns desses blocos não irão emitir uma saída para o usuário, mas será possível visualizar o tempo que demorou para ser executado ou se possui erro.


## Antes de começar: O mínimo de Python que vamos usar

Ao longo do notebook, vou explicar os recursos conforme eles aparecem. Mas existem quatro ideias que vale ter em mente desde o início:

### Variável
É um nome usado para guardar uma informação.

```python
nome = "Maria"
```

### Lista
É uma sequência de itens.

```python
frutas = ["maçã", "banana", "uva"]
```

### Dicionário
É uma estrutura com pares de **chave e valor**.

```python
pessoa = {
    "nome": "Maria",
    "idade": 30
}
```

Para acessar o nome:

```python
pessoa["nome"]
```

### Função
É um bloco de código criado para executar uma tarefa.

```python
def dobrar(numero):
    return numero * 2
```

Depois, eu posso chamar:

```python
dobrar(4)
```

e o resultado será `8`.

## Bloco 1 — Carregar os documentos

Primeiro preciso trazer os documentos para dentro do Python.

O acervo está em um arquivo JSON. Cada documento possui:

- `doc_id`: identificador;
- `tipo`: tipo do documento;
- `texto`: conteúdo pesquisável.

Exemplo:

```json
{
  "doc_id": "DOC001",
  "tipo": "Contrato",
  "texto": "Contrato para execução de obras públicas."
}
```

No próximo bloco de código, execute e indique onde está o arquivo `corpus.json`


In [1]:
import json
from google.colab import files

arquivos = files.upload()

nome_arquivo = list(arquivos.keys())[0]

documentos = json.loads(
    arquivos[nome_arquivo].decode("utf-8")
)

print("Quantidade de documentos carregados:", len(documentos))

Saving corpus.json to corpus.json
Quantidade de documentos carregados: 378


### Lendo o Python deste bloco

- `import json` carrega a biblioteca que sabe ler arquivos JSON.
- `from google.colab import files` traz a função de upload do Colab.
- `files.upload()` abre a janela para enviar o arquivo.
- `arquivos.keys()` retorna os nomes dos arquivos enviados.
- `list(...)` transforma esses nomes em uma lista.
- `[0]` pega o primeiro item dessa lista.
- `json.loads(...)` transforma o conteúdo do arquivo em dados Python.
- `len(documentos)` conta quantos documentos foram carregados.

A variável `documentos` passa a guardar todo o acervo.

In [2]:
documentos[0]

{'doc_id': 'D0072',
 'tipo': 'Notificação',
 'topico': 'recursos_humanos',
 'relevant_for': [],
 'texto': 'Foi aberto processo seletivo interno para preenchimento de vaga administrativa. Solicitamos que o presente documento seja juntado aos autos do processo administrativo correspondente. O servidor solicitou licença capacitação para participar de curso de aperfeiçoamento. Ressaltamos a importância de resposta no prazo regulamentar, conforme normativo interno vigente.'}

Aqui, `documentos[0]` significa:

> mostre o primeiro item da lista `documentos`.

Em Python, a contagem começa em zero.

## Bloco 2 — Normalizar o texto

Agora quero evitar diferenças que não deveriam afetar a busca.

Para o computador:

- `Obras`
- `obras`
- `OBRAS`

são textos diferentes.

Então começo transformando tudo em minúsculo e retirando acentos.

In [8]:
def normalizar(texto):
    texto = texto.lower()

    troca_de_letras = {
        "á": "a", "à": "a", "ã": "a", "â": "a",
        "é": "e", "ê": "e",
        "í": "i",
        "ó": "o", "ô": "o", "õ": "o",
        "ú": "u",
        "ç": "c",
    }

    for letra_com_acento in troca_de_letras:
        letra_sem_acento = troca_de_letras[letra_com_acento]
        texto = texto.replace(letra_com_acento, letra_sem_acento)

    return texto

### Lendo o Python deste bloco

- `def normalizar(texto):` cria uma função chamada `normalizar`.
- `texto` entre parênteses é a informação que a função recebe.
- `texto.lower()` transforma todas as letras em minúsculas.
- `{ ... }` cria um dicionário.
- No dicionário, `"á": "a"` significa: a chave `"á"` está associada ao valor `"a"`.
- `for letra_com_acento in troca_de_letras:` percorre cada chave do dicionário.
- `texto.replace(...)` substitui um trecho por outro.
- `return texto` devolve o resultado final da função.

Ou seja: eu entrego um texto para a função e ela me devolve uma versão padronizada.

In [7]:
print(normalizar("OBRAS E LICITAÇÃO"))

obras e licitacao


## Bloco 3 — Retirar pontuação e separar as palavras

Além de maiúsculas e acentos, quero evitar que `"obra"` e `"obra."` sejam tratados como termos diferentes.

Depois disso, separo a frase em palavras.

In [9]:
def tirar_pontuacao(texto):
    sinais = [".", ",", ";", ":", "!", "?", "(", ")", '"', "'", "-", "/"]

    for sinal in sinais:
        texto = texto.replace(sinal, " ")

    return texto


def palavras(texto):
    texto = normalizar(texto)
    texto = tirar_pontuacao(texto)

    lista_de_palavras = texto.split()

    return lista_de_palavras

### Lendo o Python deste bloco

Na função `tirar_pontuacao()`:

- `sinais = [...]` cria uma lista de sinais de pontuação.
- `for sinal in sinais:` percorre um sinal de cada vez.
- cada sinal encontrado é trocado por um espaço.

Na função `palavras()`:

- `normalizar(texto)` chama a função criada no bloco anterior.
- `tirar_pontuacao(texto)` chama outra função.
- `texto.split()` quebra o texto em partes sempre que encontra um espaço.
- o resultado vira uma lista de palavras.

Perceba que uma função pode usar outra função. Isso ajuda a dividir um problema grande em tarefas pequenas.

In [10]:
texto_teste = "Obras, Licitação e Construção!"

print(palavras(texto_teste))

['obras', 'licitacao', 'e', 'construcao']


O resultado esperado é parecido com:

```python
['obras', 'licitacao', 'e', 'construcao']
```

## Bloco 4 — Remover repetições

Em algumas etapas eu só quero saber se um termo aparece no documento.

Se uma palavra aparecer dez vezes, para essa contagem específica ela continua sendo apenas um termo presente naquele documento.

In [11]:
def sem_repetir(lista):
    lista_nova = []

    for item in lista:
        if item not in lista_nova:
            lista_nova.append(item)

    return lista_nova

### Lendo o Python deste bloco

- `lista_nova = []` cria uma lista vazia.
- `for item in lista:` percorre cada item da lista recebida.
- `if` significa **se**.
- `item not in lista_nova` verifica se aquele item ainda não está na nova lista.
- `append(item)` adiciona o item ao final da lista.
- `return lista_nova` devolve a lista sem repetições.

A lógica em português seria:

> para cada item da lista, se ele ainda não estiver na lista nova, adicione.

In [12]:
exemplo = ["obra", "obra", "obra", "contrato"]

print(sem_repetir(exemplo))

['obra', 'contrato']


## Bloco 5 — Criar n-gramas de caracteres

Parte do acervo pode vir de OCR.

OCR transforma uma imagem digitalizada em texto, mas pode cometer erros.

Se uma palavra como:

`contrato`

virar:

`contralo`

uma comparação por palavra completa pode falhar.

Por isso, também quebro o texto em pedaços menores.

In [13]:
def n_gramas_de_caractere(texto, tamanho=4):
    texto = normalizar(texto)
    texto = texto.replace(" ", "")

    pedacos = []
    posicao = 0

    while posicao + tamanho <= len(texto):
        pedaco = texto[posicao:posicao + tamanho]
        pedacos.append(pedaco)

        posicao = posicao + 1

    return pedacos

### Lendo o Python deste bloco

- `tamanho=4` define um valor padrão. Se eu não informar outro tamanho, a função usa 4.
- `texto.replace(" ", "")` remove os espaços.
- `pedacos = []` cria a lista onde os n-gramas serão guardados.
- `posicao = 0` define onde a leitura começa.
- `while` significa **enquanto**.
- `len(texto)` conta quantos caracteres o texto possui.
- `texto[posicao:posicao + tamanho]` recorta uma parte do texto.
- `append()` adiciona o pedaço na lista.
- `posicao = posicao + 1` avança uma posição.

O `while` repete esse processo até chegar ao fim do texto.

In [ ]:
print(n_gramas_de_caractere("obras"))

Para `"obras"`, o resultado será:

```python
['obra', 'bras']
```

## Bloco 6 — Contar em quantos documentos cada termo aparece

Agora quero descobrir quais termos são muito comuns e quais são mais raros.

A ideia é simples:

- termo muito comum ajuda pouco;
- termo raro ajuda mais a diferenciar documentos.

In [14]:
def contar_em_quantos_documentos_cada_termo_aparece(documentos):
    quantos_docs_tem_a_palavra = {}
    quantos_docs_tem_o_ngrama = {}

    for documento in documentos:

        lista_de_palavras = palavras(documento["texto"])
        palavras_unicas_do_doc = sem_repetir(lista_de_palavras)

        for palavra in palavras_unicas_do_doc:

            if palavra in quantos_docs_tem_a_palavra:
                quantos_docs_tem_a_palavra[palavra] = (
                    quantos_docs_tem_a_palavra[palavra] + 1
                )
            else:
                quantos_docs_tem_a_palavra[palavra] = 1

        lista_de_ngramas = n_gramas_de_caractere(documento["texto"])
        ngramas_unicos_do_doc = sem_repetir(lista_de_ngramas)

        for ngrama in ngramas_unicos_do_doc:

            if ngrama in quantos_docs_tem_o_ngrama:
                quantos_docs_tem_o_ngrama[ngrama] = (
                    quantos_docs_tem_o_ngrama[ngrama] + 1
                )
            else:
                quantos_docs_tem_o_ngrama[ngrama] = 1

    return quantos_docs_tem_a_palavra, quantos_docs_tem_o_ngrama

### Lendo o Python deste bloco

Aqui aparecem algumas ideias novas.

- `quantos_docs_tem_a_palavra = {}` cria um dicionário vazio.
- `for documento in documentos:` percorre o acervo documento por documento.
- `documento["texto"]` acessa o texto daquele documento.
- `if palavra in quantos_docs_tem_a_palavra:` verifica se a palavra já existe como chave no dicionário.
- Se existir, somo `+ 1`.
- `else` significa **caso contrário**.
- Se ainda não existir, começo a contagem com `1`.

Exemplo:

```python
{
    "contrato": 12,
    "licitacao": 5
}
```

Isso quer dizer que `"contrato"` apareceu em 12 documentos e `"licitacao"` em 5.

O mesmo raciocínio é repetido para os n-gramas.

No final:

```python
return quantos_docs_tem_a_palavra, quantos_docs_tem_o_ngrama
```

devolve **dois resultados ao mesmo tempo**.

## Bloco 7 — Dar mais peso para termos raros

Agora transformo a contagem em um peso.

Quanto menos documentos possuem um termo, maior será o peso dele.

In [15]:
def peso_por_raridade(
    termo,
    contagem_de_documentos,
    total_de_documentos
):

    if termo in contagem_de_documentos:
        quantos_docs_tem = contagem_de_documentos[termo]
    else:
        quantos_docs_tem = 0

    peso = total_de_documentos / (quantos_docs_tem + 1)

    return peso

### Lendo o Python deste bloco

- a função recebe três informações: `termo`, `contagem_de_documentos` e `total_de_documentos`;
- `if termo in contagem_de_documentos` verifica se o termo existe no dicionário;
- `contagem_de_documentos[termo]` pega o valor associado àquela chave;
- `/` faz divisão;
- `+ 1` evita uma divisão por zero;
- `return peso` devolve o resultado calculado.

É uma regra simples para representar a mesma ideia de raridade.

Agora preciso somar os pesos dos termos que aparecem tanto na pergunta quanto no documento.

In [16]:
def soma_dos_pesos_em_comum(
    lista_a,
    lista_b,
    contagem_de_documentos,
    total_de_documentos
):

    lista_a_unica = sem_repetir(lista_a)
    lista_b_unica = sem_repetir(lista_b)

    soma = 0

    for item in lista_a_unica:

        if item in lista_b_unica:
            soma = soma + peso_por_raridade(
                item,
                contagem_de_documentos,
                total_de_documentos
            )

    return soma

### Lendo o Python deste bloco

- `soma = 0` cria um acumulador começando em zero.
- `for item in lista_a_unica:` percorre a primeira lista.
- `if item in lista_b_unica:` verifica se o mesmo item também existe na segunda lista.
- se existir, chamo `peso_por_raridade(...)`.
- o valor retornado é adicionado à variável `soma`.

Ou seja:

> para cada termo em comum, calculo seu peso e acrescento ao total.

## Bloco 8 — Calcular o placar de um documento

Agora junto tudo o que foi construído até aqui.

Quero responder:

> quanto este documento combina com a pergunta do usuário?

In [17]:
def pontuar_documento(
    pergunta,
    texto_do_documento,
    contagem_palavra,
    contagem_ngrama,
    total_de_documentos
):

    palavras_da_pergunta = palavras(pergunta)
    palavras_do_documento = palavras(texto_do_documento)

    pontos_de_palavra = soma_dos_pesos_em_comum(
        palavras_da_pergunta,
        palavras_do_documento,
        contagem_palavra,
        total_de_documentos
    )

    ngramas_da_pergunta = n_gramas_de_caractere(pergunta)
    ngramas_do_documento = n_gramas_de_caractere(texto_do_documento)

    pontos_de_ngrama = soma_dos_pesos_em_comum(
        ngramas_da_pergunta,
        ngramas_do_documento,
        contagem_ngrama,
        total_de_documentos
    )

    placar_final = pontos_de_palavra + (pontos_de_ngrama * 0.3)

    return placar_final

### Lendo o Python deste bloco

Esta função não cria um conceito novo de Python. Ela **combina funções anteriores**.

Primeiro:

```python
palavras(pergunta)
```

transforma a pergunta em palavras.

Depois:

```python
soma_dos_pesos_em_comum(...)
```

calcula os pontos das palavras coincidentes.

O mesmo acontece com os n-gramas.

No final:

```python
placar_final = pontos_de_palavra + (pontos_de_ngrama * 0.3)
```

significa:

- peso completo para palavras;
- 30% do peso calculado para n-gramas.

O `0.3` é apenas um parâmetro inicial do protótipo.

## Bloco 9 — Ordenar do maior placar para o menor

Agora cada documento pode ter uma nota.

Preciso colocar os melhores no topo.

In [18]:
def ordenar_do_maior_para_o_menor(lista_de_pares):

    restantes = []

    for par in lista_de_pares:
        restantes.append(par)

    ordenado = []

    while len(restantes) > 0:

        indice_do_maior = 0

        for i in range(len(restantes)):

            if restantes[i][0] > restantes[indice_do_maior][0]:
                indice_do_maior = i

        ordenado.append(restantes[indice_do_maior])
        restantes.pop(indice_do_maior)

    return ordenado

### Lendo o Python deste bloco

Aqui aparecem alguns recursos novos.

Cada resultado é guardado como um par:

```python
[placar, documento]
```

Então:

- `restantes[i][0]` acessa o placar do item atual;
- `range(len(restantes))` gera os índices da lista;
- `indice_do_maior` guarda a posição do maior placar encontrado;
- `pop(indice_do_maior)` remove esse item da lista;
- `append(...)` coloca esse item na lista ordenada.

O `while` repete até não restar nenhum item.

É uma ordenação feita manualmente para deixar a lógica visível.

## Bloco 10 — Criar a função de busca

Agora junto todas as etapas em uma única função.

Para cada documento:

1. calculo o placar;
2. guardo quem teve placar maior que zero;
3. ordeno;
4. devolvo apenas os primeiros resultados.

In [19]:
def buscar(
    pergunta,
    documentos,
    contagem_palavra,
    contagem_ngrama,
    quantos_resultados=5
):

    total_de_documentos = len(documentos)
    resultados = []

    for documento in documentos:

        placar = pontuar_documento(
            pergunta,
            documento["texto"],
            contagem_palavra,
            contagem_ngrama,
            total_de_documentos
        )

        if placar > 0:
            resultados.append([placar, documento])

    resultados_ordenados = ordenar_do_maior_para_o_menor(resultados)

    return resultados_ordenados[:quantos_resultados]

### Lendo o Python deste bloco

- `quantos_resultados=5` define que, por padrão, quero os cinco melhores.
- `len(documentos)` conta o tamanho do acervo.
- `resultados = []` cria a lista onde os candidatos serão guardados.
- `for documento in documentos:` percorre todo o acervo.
- `pontuar_documento(...)` chama a função criada anteriormente.
- `if placar > 0:` guarda apenas documentos com alguma correspondência.
- `[placar, documento]` cria um par contendo a nota e o próprio documento.
- `resultados_ordenados[:quantos_resultados]` pega apenas o começo da lista.

Se `quantos_resultados` for 5, `[:5]` significa:

> pegue os cinco primeiros itens.

## Bloco 11 — Construir o índice antes das buscas

Antes de receber uma pergunta, preparo as contagens do acervo.

In [20]:
contagem_palavra, contagem_ngrama = (
    contar_em_quantos_documentos_cada_termo_aparece(documentos)
)

print("Índice construído.")
print("Documentos:", len(documentos))
print("Palavras indexadas:", len(contagem_palavra))
print("N-gramas indexados:", len(contagem_ngrama))

Índice construído.
Documentos: 378
Palavras indexadas: 386
N-gramas indexados: 2542


### Lendo o Python deste bloco

A função retorna dois valores:

```python
return quantos_docs_tem_a_palavra, quantos_docs_tem_o_ngrama
```

Por isso consigo recebê-los diretamente em duas variáveis:

```python
contagem_palavra, contagem_ngrama = ...
```

Essa etapa é feita antes da busca para não recalcular tudo a cada pergunta.

## Bloco 12 — Fazer uma pergunta

Agora o protótipo pode ser usado.

In [21]:
pergunta = input("O que você quer procurar? ")

resultados = buscar(
    pergunta,
    documentos,
    contagem_palavra,
    contagem_ngrama,
    quantos_resultados=5
)

if len(resultados) == 0:

    print("Nenhum documento encontrado.")

else:

    print("\nResultados encontrados:\n")

    posicao = 1

    for par in resultados:

        placar = par[0]
        documento = par[1]

        print(
            str(posicao) + ".",
            "[" + documento["doc_id"] + "]",
            documento["tipo"],
            "| placar:",
            round(placar, 1)
        )

        trecho = documento["texto"][:200]

        if len(documento["texto"]) > 200:
            trecho = trecho + "..."

        print(trecho)
        print()

        posicao = posicao + 1

O que você quer procurar? Saude teve desvio de dinheiro

Resultados encontrados:

1. [D0367] Relatório de Fiscalização | placar: 303.5
A prestação de contas do posto de saúde apresentou inconsistências entre os valores repassados e os insumos efetivamente recebidos pela unidade. Este documento foi produzido no âmbito das atividades o...

2. [D0366] Memorando | placar: 57.8
Relato anônimo indica que recursos destinados à compra de medicamentos teriam sido direcionados a finalidade distinta da prevista no convênio com o hospital. Solicitamos que o presente documento seja ...

3. [D0374] Ata de Reunião | placar: 57.8
Em atenção ao ofício anterior, informamos que o assunto está em análise pela equipe responsável. Registre-se que as informações aqui prestadas têm caráter preliminar e poderão ser complementadas. O re...

4. [D0056] Memorando | placar: 18.5
Registre-se que as informações aqui prestadas têm caráter preliminar e poderão ser complementadas. Encaminha-se cópia do expediente para 

### Lendo o Python deste bloco

- `input(...)` espera uma informação digitada pelo usuário.
- essa informação fica guardada na variável `pergunta`.
- `buscar(...)` chama a função principal.
- `if len(resultados) == 0` verifica se a lista está vazia.
- `==` significa **é igual a**.
- `else` executa o outro caminho caso existam resultados.
- `par[0]` pega o placar.
- `par[1]` pega o documento.
- `documento["doc_id"]` acessa o identificador.
- `documento["tipo"]` acessa o tipo.
- `round(placar, 1)` arredonda o número para uma casa decimal.
- `documento["texto"][:200]` pega apenas os primeiros 200 caracteres.
- `posicao = posicao + 1` aumenta a posição do ranking.

Aqui aparece o fluxo completo:

**pergunta → busca → pontuação → ordenação → apresentação**

# O que este protótipo demonstra

O objetivo deste código não é construir uma solução final de IA.

Ele demonstra o menor fluxo funcional da busca dirigida:

**o usuário informa o que procura → o sistema compara essa pergunta com o acervo → prioriza documentos → o analista valida**

Depois que o conceito estiver validado, várias etapas podem ser simplificadas com bibliotecas de ML do Python.

# Sugestão de melhoria para uma próxima versão

| Parte do notebook | O que foi implementado manualmente | Alternativa | Biblioteca / recurso |
|---|---|---|---|
| **Bloco 2 e 3 — Preparação do texto** | minúsculas, pontuação, separação em palavras | automatizar boa parte da vetorização e pré-processamento | `TfidfVectorizer` — scikit-learn |
| **Bloco 4 — Remover repetições** | `for` + `if` para construir uma nova lista | remover duplicados diretamente | `set()` — Python |
| **Bloco 5 — N-gramas** | recorte manual do texto com `while` | gerar n-gramas automaticamente | `TfidfVectorizer(analyzer="char", ngram_range=...)` |
| **Bloco 6 — Frequência documental** | dicionários e laços para contar documentos por termo | calcular frequência documental automaticamente | `TfidfVectorizer` |
| **Bloco 7 — Peso por raridade** | fórmula própria de peso | utilizar TF-IDF | `TfidfVectorizer` |
| **Bloco 8 — Pontuação** | soma manual de palavras e n-gramas coincidentes | comparar vetores numericamente | `cosine_similarity` — scikit-learn |
| **Bloco 9 — Ordenação** | busca manual do maior valor | ordenar diretamente | `sorted()` — Python |
| **Bloco 10 — Busca** | combinação manual de pontuação, filtro e ranking | vetorizar corpus e consulta e calcular similaridade | `TfidfVectorizer` + `cosine_similarity` |
